# 2026 Spring "EE50045, QU50001: Introductory Quantum Mechanics"

###### Update history
###### 2021. 01. 01 : Hyeonwoo Yeo, KAIST Electrical Engineering, Initial implementation of TDSE code. (v1)
###### 2024. 06. 26 : Minsu Jeong,  KAIST Electrical Engineering, minor revision and clean up
###### 2026. 03. 12 : Minsu Jeong,  KAIST Electrical Engineering, implemented the Fast Fourier Transform (FFT) algorithm and improved the propagation routine. (v2)
###### 2026. 03. 16 : Minsu Jeong & Jeongwon Lee,  KAIST Electrical Engineering, updated k-space grid formulation.


###### ref
###### - Neaman, Semiconductor Physics and Devices Basic Principles, 4ed (McGraw-Hill, 2012), p31
###### - John R. Hiller, Quantum Mechanics Simulations The Consortium for upper-Level Physics Software, (John Wiley & Sons)
###### - Stamp, A. P., & McIntosh, G. C. (1996). A time‐dependent study of resonant tunneling through a double barrier. American Journal of Physics, 64(3), 264-276.

# Time-Dependent Schrödinger Equation (TDSE)

---

## Basic Formulation

- The TDSE describes how a quantum state evolves in time:
  $$
  i\hbar\,\partial_t\psi(x,t)=\hat{H}\psi(x,t).
  $$

- For one particle in 1D:
  $$
  \hat{H}=\hat{T}+\hat{V}= -\frac{\hbar^2}{2m}\frac{\partial^2}{\partial x^2}+V(x).
  $$

- Probability density and normalization:
  $$
  \rho(x,t)=|\psi(x,t)|^2,\qquad \int |\psi(x,t)|^2\,dx = 1.
  $$


## Initial Wave Packet

- The simulation starts from a Gaussian wave packet centered at $x_0$ with mean wave number $k_0$.

- In momentum space:
  $$
  \tilde\psi(k,0) \propto \exp\!\left[-\frac{(k-k_0)^2}{2\sigma_k^2}\right]e^{-ikx_0},\qquad \sigma_k\approx\frac{1}{\sigma_x}.
  $$

- The real-space initial state is obtained by an inverse FFT and then normalized.

- In the code, this corresponds to constructing `wave_k`, applying `ifft`, and normalizing `gridwave`.

## Grid Discretization and Fourier Grid

- The spatial domain is sampled on a uniform grid:
  $$
  x_j = x_{\min}+j\,\Delta x,\qquad j=0,1,\dots,N_x-1.
  $$

- The reciprocal grid used by FFT is:
  $$
  k_n = 2\pi\,\mathrm{fftfreq}(N_x,\Delta x).
  $$

- Atomic units are used internally for propagation (Hartree and Bohr), while plotting often converts quantities to eV and Å.

## Time Propagation

- The exact one-step propagation is formally written as
  $$
  \psi(t+\Delta t)=e^{-\frac{i}{\hbar}(\hat{T}+\hat{V})\Delta t}\psi(t).
  $$

- In general, the kinetic operator $\hat{T}$ and the potential operator $\hat{V}$ do not commute, so the full exponential is not evaluated directly at each step.

- In this notebook, the time evolution is approximated by the split-operator method:
  $$
  e^{-i(\hat{T}+\hat{V})\Delta t}
  \approx
  e^{-i\hat{V}\Delta t/2}\,e^{-i\hat{T}\Delta t}\,e^{-i\hat{V}\Delta t/2}.
  $$

- This is efficient because the potential term is diagonal in real space, while the kinetic term becomes diagonal in Fourier ($k$) space.

- After transforming to Fourier space, the kinetic propagation reduces to a simple phase multiplication. In atomic units,
  $$
  \hat{T}=\frac{\hat{p}^2}{2}
  \qquad\Rightarrow\qquad
  e^{-i\hat{T}\Delta t}\to e^{-ik^2\Delta t/2}.
  $$

- Therefore, one propagation step is carried out as follows:
  1. Multiply the wavefunction by the half-step potential phase $e^{-iV(x)\Delta t/2}$.
  2. Apply FFT to move from $x$ space to $k$ space.
  3. Multiply by the kinetic phase $e^{-ik^2\Delta t/2}$.
  4. Apply inverse FFT to return to real space.
  5. Multiply once more by the half-step potential phase.

- In the code, this corresponds to `_propagate_fft_step(...)`, where `phase_v_half`, `fft_backend(...)`, `phase_t`, and `ifft_backend(...)` are applied in sequence.

- After each time step, the notebook also records observables such as probability density, reflection, transmission, and downsampled arrays for visualization. This is handled in `_record_observables(...)`.

- For the harmonic-well linear-combination mode, the code may propagate the expansion coefficients in the eigenstate basis instead of using the FFT step directly. This is a special branch, while the main propagation framework of the notebook is the FFT-based split-operator method.

In [8]:
# install required libraries (run once)
import sys
print("Python executable:", sys.executable)

# !{sys.executable} -m pip install -q --upgrade pip
# !{sys.executable} -m pip install -q --upgrade numpy scipy matplotlib ipython ipywidgets widgetsnbextension jupyterlab_widgets plotly nbformat tqdm

import ipywidgets as widgets
print("ipywidgets:", widgets.__version__)

import nbformat
print("nbformat:", nbformat.__version__)


Python executable: /home/minsu/anaconda3/envs/gemini/bin/python
ipywidgets: 8.1.8
nbformat: 5.10.4


In [9]:
###############################################################################
# Global Constants, Imports, and Unit Conversions
###############################################################################

import numpy as np
from dataclasses import dataclass
from scipy import fft as sp_fft
from scipy import linalg as sc_lin
from scipy import constants as sc_const
from tqdm.auto import tqdm
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display
from ipywidgets import interact_manual
import ipywidgets as widgets

# Set numpy print options for debugging (optional)
np.set_printoptions(precision=6, suppress=True)

@dataclass(frozen=True)
class UnitSystem:
    ang2bohr: float
    har2ev: float
    au2fs_time: float


@dataclass
class SimulationConfig:
    boxL_ang: float = 500.0
    ngridx_max: int = 70001
    dx_target_ang: float = 0.1
    barrier_distance_ang: int = 3
    dt_fs: float = 0.1
    simulation_time_fs_default: float = 15.0


def _build_unit_system():
    bohr_radius_m = sc_const.physical_constants["Bohr radius"][0]
    angstrom_m = sc_const.angstrom
    return UnitSystem(
        ang2bohr=angstrom_m / bohr_radius_m,
        har2ev=sc_const.physical_constants["Hartree energy in eV"][0],
        au2fs_time=sc_const.physical_constants["atomic unit of time"][0] * 1e15,
    )


UNITS = _build_unit_system()
SIM_CFG = SimulationConfig()

# Unit aliases used by solver/plot functions in later cells
ang2bohr = UNITS.ang2bohr
har2ev = UNITS.har2ev
au2fs_time = UNITS.au2fs_time

In [10]:
###############################################################################
# FFT, Grid, and Basis Helper Functions
###############################################################################

def fftfreq_backend(n, d):
    return sp_fft.fftfreq(n, d=d)


def fft_backend(arr):
    return sp_fft.fft(arr, workers=-1)


def ifft_backend(arr):
    return sp_fft.ifft(arr, workers=-1)


def solve_low_eigenstates(pot_grid, dx_local, nb):
    """
    Solve low-lying eigenstates of finite-difference Hamiltonian.
    """
    if pot_grid is None:
        return None, None

    nint = len(pot_grid) - 2
    if nint < 2:
        return None, None

    nb = max(1, min(int(nb), nint))
    inv_dx2 = 1.0 / (dx_local * dx_local)

    diag = inv_dx2 + pot_grid[1:-1]
    off = (-0.5 * inv_dx2) * np.ones(nint - 1, dtype=np.float64)

    evals, evecs = sc_lin.eigh_tridiagonal(
        diag,
        off,
        select='i',
        select_range=(0, nb - 1),
        check_finite=False
    )

    basis = np.zeros((len(pot_grid), nb), dtype=np.complex128)
    basis[1:-1, :] = evecs / np.sqrt(dx_local)

    return np.asarray(evals, dtype=np.float64), basis

def get_save_stride(ngridx, dx_target_ang=0.1, x_resolution="mid"):
    """
    Set save stride for space. Time save stride is always 1 (all simulation steps saved).
    """
    t_stride = 1
    base_nx = max(1, int(round(360.0 / max(dx_target_ang, 1e-12))))

    if x_resolution == "low":
        target_saved_nx = max(1, int(round(0.5 * base_nx)))
    elif x_resolution == "high":
        target_saved_nx = max(1, int(round(5.0 * base_nx)))
    elif x_resolution == "full":
        target_saved_nx = max(1, int(ngridx))
    else:
        target_saved_nx = base_nx

    target_saved_nx = min(max(1, int(ngridx)), target_saved_nx)

    print(f"Full # of grid (position) = {max(1, int(ngridx))}")
    print(f"Plotted # of grid (position) = {target_saved_nx}")

    x_stride = max(1, int(round(float(ngridx) / float(target_saved_nx))))
    return t_stride, x_stride

def set_grid_by_dispersion(dispersion_gaussian_log10, dx_target_ang, ngridx_max):
    """
    Set box length and spatial grid from dispersion.

    Rules:
    - Keep box length driven by dispersion.
    - Use dx_target_ang when possible.
    - If ngridx is capped by ngridx_max, keep box length and relax effective dx.

    input:
        int         dispersion_gaussian_log10    log10 Gaussian dispersion in real-space [Angstrom]
        float       dx_target_ang                Target grid spacing [Angstrom]
        int         ngridx_max                   Maximum number of spatial grid points
    output:
        float       sigma_x_ang                  Gaussian dispersion in real-space [Angstrom]
        float       boxL                         Box length [Angstrom]
        int         ngridx                       Number of spatial grid points
        float       dx_ang_effective             Effective grid spacing [Angstrom]
    """
    sigma_x_ang = 10.0**int(dispersion_gaussian_log10)
    boxL_target = max(360.0, 70.0 * sigma_x_ang)
    max_boxL = 70.0 * (10.0**4)
    boxL = float(min(boxL_target, max_boxL))

    dx_target_ang = max(float(dx_target_ang), 1e-12)
    ngridx_target = max(201, int(round(boxL / dx_target_ang)))
    ngridx = max(201, min(ngridx_target, int(ngridx_max)))

    dx_ang_effective = boxL / float(ngridx)

    if ngridx < ngridx_target:
        print(
            f"[grid] ngridx capped at {ngridx} (target={ngridx_target}); "
            f"effective dx={dx_ang_effective:.4f} Angstrom (target={dx_target_ang:.4f})"
        )

    return sigma_x_ang, boxL, ngridx, dx_ang_effective


def set_packet_center(boxL, sigma_x_ang):
    """
    Set packet center.

    input:
        float       boxL            Box length [Angstrom]
        float       sigma_x_ang     Gaussian dispersion in real-space [Angstrom]
    output:
        float       packet_center   Initial center of wave packet (fraction of box)
    """
    return 0.30

In [11]:
###############################################################################
# Initial Wave Packet and Potential Functions
###############################################################################

def make_wave(boxL, ngridx, pot_shape, sigma_x, k, lpacket, ncombistates, packet_center, pot_grid=None):
    """
    Construct initial wave packet and return it on a spatial grid.

    input:
        int         boxL            Box size
        int         ngridx          Number of spatial grid points
        int         pot_shape       Shape of the potential
        float       sigma_x         Gaussian dispersion in real-space
        float       k               Wave vector (momentum)
        int         lpacket         0: wave packet, 1: linear combination, 2: coherent
        int         ncombistates    Number of basis states (linear combination)
        float       packet_center   Initial center of wave packet (fraction of box)
        np.array    pot_grid        Potential on grid
    output:
        np.array    gridwave        Initial wave packet
        np.array    eigvals_lin     Eigenvalues for linear-combination basis
        np.array    basis_lin       Basis vectors for linear-combination basis
        np.array    coeff_lin       Coefficients for linear-combination basis
    """
    x0 = packet_center * boxL
    dx_local = np.float64(boxL / ngridx)

    gridwave = np.zeros(ngridx, np.complex128)
    eigvals_lin = None
    basis_lin = None
    coeff_lin = None

    # Gaussian packet in k-space
    kgrid = 2.0 * np.pi * fftfreq_backend(ngridx, d=dx_local)
    sigma_k = 1.0 / max(float(sigma_x), 1e-12)
    wave_k = np.exp(-0.5 * ((kgrid - k) / sigma_k)**2).astype(np.complex128)
    wave_k *= np.exp(-1j * kgrid * x0)

    gauss_wave = ifft_backend(wave_k)
    gauss_norm = sc_lin.norm(gauss_wave)
    if gauss_norm > 1e-14:
        gauss_wave /= gauss_norm

    # Wave packet
    if lpacket == 0:
        gridwave = gauss_wave.copy()

    # Linear combination
    elif lpacket == 1:
        nb = max(1, min(int(ncombistates), ngridx - 2))
        eigvals_lin, basis_lin = solve_low_eigenstates(pot_grid, dx_local, nb)
        if (eigvals_lin is not None) and (basis_lin is not None):
            # Equal-weight superposition of the first nb eigenstates
            coeff_lin = np.full(nb, 1.0 / np.sqrt(nb), dtype=np.complex128)
            gridwave = basis_lin @ coeff_lin
        else:
            gridwave = gauss_wave.copy()

    if coeff_lin is None:
        norm = sc_lin.norm(gridwave)
        if norm < 1e-14:
            gridwave[(3 * ngridx) // 10:(5 * ngridx) // 10] = 1.0
            norm = sc_lin.norm(gridwave)
        gridwave /= norm

    return gridwave, eigvals_lin, basis_lin, coeff_lin

def _centered_window(center, width_points, ngridx):
    half = max(1, width_points) // 2
    i0 = max(1, center - half)
    i1 = min(ngridx - 1, i0 + max(1, width_points))
    if i1 <= i0:
        i1 = min(ngridx - 1, i0 + 1)
    return i0, i1


def potential(ngridx, boxL, pot_shape, pot_height_eV, barrier_distance, curvature=0.5, width=6.0, slope=1.0):
    """
    Set the shape of potential V and return the potential on a grid.

    input:
        int         ngridx              Number of spatial grid points
        float       boxL                Box length [Angstrom]
        int         pot_shape           Shape of potential
        int         pot_height_eV       Height of the potential barrier (eV)
        int         barrier_distance    Distance between two barriers (Angstrom)
        float       curvature           Curvature for harmonic potential
        float       width               Width parameter (Angstrom)
        float       slope               Slope parameter
    output:
        np.array    pot_grid            Potential on the grid
    """
    pot_height_har = pot_height_eV / har2ev
    pot_grid = np.zeros(ngridx)
    pot_grid[0] = 100000000
    pot_grid[ngridx - 1] = 100000000

    dx_ang = max(float(boxL) / float(ngridx), 1e-12)
    width_points = max(1, int(width / dx_ang))
    center = ngridx // 2

    # Step potential
    if pot_shape == 1:
        pot_grid[center:ngridx - 1] = pot_height_har

    # Single potential barrier
    elif pot_shape == 2:
        i0, i1 = _centered_window(center, width_points, ngridx)
        pot_grid[i0:i1] = pot_height_har

    # Double potential barrier
    elif pot_shape == 3:
        gap_points = max(1, int(barrier_distance / dx_ang))
        l0 = max(1, center - gap_points // 2 - width_points)
        l1 = max(l0 + 1, center - gap_points // 2)
        r0 = min(ngridx - 2, center + gap_points // 2)
        r1 = min(ngridx - 1, r0 + width_points)
        pot_grid[l0:l1] = pot_height_har
        pot_grid[r0:r1] = pot_height_har

    # Square potential well
    elif pot_shape == 4:
        pot_grid[1:ngridx - 1] = pot_height_har
        i0, i1 = _centered_window(center, width_points, ngridx)
        pot_grid[i0:i1] = 0.0

    # Harmonic potential well
    elif pot_shape == 5:
        c = max(0.1, float(curvature))
        x_ang = (np.arange(ngridx) - center) * dx_ang
        x_scale = max(np.max(np.abs(x_ang)), 1e-12)
        pot_grid[1:ngridx - 1] = ((x_ang[1:ngridx - 1] / x_scale)**2) * pot_height_har / c

    # Triangular potential well
    elif pot_shape == 6:
        s = max(0.1, float(slope))
        start = int(ngridx * 0.4)
        pot_grid[1:ngridx - 1] = pot_height_har
        idx = np.arange(start, ngridx - 1)
        ramp = (pot_height_har * (idx - start) / max(ngridx - start, 1)) * s
        pot_grid[idx] = np.minimum(pot_height_har, ramp)

    return pot_grid


def _transmission_start_index(pot_shape, ngridx, boxL, width, barrier_distance):
    dx_ang = max(float(boxL) / float(ngridx), 1e-12)
    width_points = max(1, int(width / dx_ang))
    center = ngridx // 2

    if pot_shape == 2:
        return min(ngridx - 1, center + width_points // 2)
    if pot_shape == 3:
        gap_points = max(1, int(barrier_distance / dx_ang))
        return min(ngridx - 1, center + gap_points // 2 + width_points)
    return center

In [12]:
###############################################################################
# Time Propagation and TDSE Solver
###############################################################################

def _record_observables(psi, prob_re, prob_im, trans_start, reflection, transmission, it,
                        wave_ds, psi_ds, kprob_ds, time_fs_ds, x_stride, k_stride, t_stride, dt):
    np.multiply(psi.real, psi.real, out=prob_re)
    np.multiply(psi.imag, psi.imag, out=prob_im)
    np.add(prob_re, prob_im, out=prob_re)

    total_prob = np.sum(prob_re)
    left_prob = np.sum(prob_re[:trans_start])
    reflection[it] = left_prob
    transmission[it] = total_prob - left_prob

    if (it % t_stride) == 0:
        isave = it // t_stride
        wave_ds[isave, :] = prob_re[::x_stride]
        psi_ds[isave, :] = psi[::x_stride]
        psi_k_full = np.fft.fftshift(np.fft.fft(psi))
        kprob_ds[isave, :] = (np.abs(psi_k_full)**2)[::k_stride]
        # Snapshot is recorded after one propagation step.
        time_fs_ds[isave] = np.float32((it + 1) * dt * au2fs_time)


def _propagate_fft_step(psi, psi_k, phase_v_half, phase_t):
    psi *= phase_v_half
    psi_k[:] = fft_backend(psi)
    psi_k *= phase_t
    psi[:] = ifft_backend(psi_k)
    psi *= phase_v_half


def time_dep_schr_eq(
    pot_shape,
    pot_height_eV,
    ewave,
    dispersion_gaussian_log10,
    ngridt,
    lpacket,
    ncombistates,
    packet_center,
    barrier_distance,
    curvature,
    width,
    slope,
    boxL,
    boxL_bohr,
    ngridx,
    dx,
    dt,
    x_resolution="mid",
    dx_target_ang=0.1,
    show_progress=True,
):
    """
    Solve the time-dependent Schrodinger equation using Fourier transform propagation.
    """
    k = np.sqrt(ewave * 2.0 / har2ev)
    sigma_x_ang = 10.0**int(dispersion_gaussian_log10)
    sigma_x = max(sigma_x_ang, 1e-12) * ang2bohr

    pot = potential(ngridx, boxL, pot_shape, pot_height_eV, barrier_distance, curvature, width, slope)
    psi, eigvals_lin, basis_lin, coeff_lin = make_wave(
        boxL_bohr, ngridx, pot_shape, sigma_x, k, lpacket, ncombistates, packet_center, pot
    )

    kgrid = 2.0 * np.pi * fftfreq_backend(ngridx, d=dx)
    phase_v_half = np.exp(-0.5j * pot * dt)
    phase_t = np.exp(-0.5j * (kgrid**2) * dt)

    t_stride, x_stride = get_save_stride(ngridx, dx_target_ang, x_resolution)
    nsave_t = (ngridt + t_stride - 1) // t_stride

    position = np.linspace(-0.5 * boxL_bohr, 0.5 * boxL_bohr, ngridx, endpoint=False) / ang2bohr
    dx_ang_full = float(dx / ang2bohr)
    kgrid_ang_full = 2.0 * np.pi * np.fft.fftshift(np.fft.fftfreq(ngridx, d=dx_ang_full))
    nk_target = 4096
    k_stride = max(1, int(np.ceil(float(ngridx) / float(nk_target))))
    k_axis_ds = np.asarray(kgrid_ang_full[::k_stride], dtype=np.float32)

    x_ang_ds = np.asarray(position[::x_stride], dtype=np.float32)
    wave_ds = np.zeros((nsave_t, len(x_ang_ds)), dtype=np.float32)
    psi_ds = np.zeros((nsave_t, len(x_ang_ds)), dtype=np.complex64)
    kprob_ds = np.zeros((nsave_t, len(k_axis_ds)), dtype=np.float32)
    time_fs_ds = np.zeros(nsave_t, dtype=np.float32)

    reflection = np.zeros(ngridt)
    transmission = np.zeros(ngridt)
    trans_start = _transmission_start_index(pot_shape, ngridx, boxL, width, barrier_distance)

    psi_k = np.empty(ngridx, dtype=np.complex128)
    prob_re = np.empty(ngridx, dtype=np.float64)
    prob_im = np.empty(ngridx, dtype=np.float64)

    use_eigen_propagation = (
        lpacket == 1 and
        (eigvals_lin is not None) and
        (basis_lin is not None) and
        (coeff_lin is not None)
    )

    if use_eigen_propagation:
        phase_eig_step = np.exp(-1j * eigvals_lin * dt)
        coeff_t = coeff_lin.copy()

    step_iter = tqdm(range(ngridt), total=ngridt, desc="TDSE propagation", leave=True) if show_progress else range(ngridt)
    for it in step_iter:
        if use_eigen_propagation:
            coeff_t *= phase_eig_step
            psi[:] = basis_lin @ coeff_t
        else:
            _propagate_fft_step(psi, psi_k, phase_v_half, phase_t)

        _record_observables(
            psi, prob_re, prob_im, trans_start, reflection, transmission, it,
            wave_ds, psi_ds, kprob_ds, time_fs_ds, x_stride, k_stride, t_stride, dt
        )

    return (
        pot, wave_ds, psi_ds, kprob_ds, time_fs_ds, x_ang_ds, k_axis_ds,
        t_stride, x_stride, k_stride, reflection, transmission
    )

In [13]:
###############################################################################
# Animation and Visualization Functions
###############################################################################

def _animate_args(duration_ms, fromcurrent=True):
    return {
        "fromcurrent": fromcurrent,
        "frame": {"duration": duration_ms, "redraw": True},
        "transition": {"duration": 0}
    }


def _kspace_probability(psi_ds, dx_ang):
    psi_k = np.fft.fftshift(np.fft.fft(psi_ds, axis=1), axes=1)
    kgrid = 2.0 * np.pi * np.fft.fftshift(np.fft.fftfreq(psi_ds.shape[1], d=dx_ang))
    prob_k = np.abs(psi_k)**2
    return kgrid, prob_k


def _frame_indices_from_time_resolution(time_fs, time_resolution_log10_s):
    nframe = len(time_fs)
    if nframe <= 1:
        return np.arange(nframe, dtype=np.int64)

    dt_min_fs = (10.0 ** float(time_resolution_log10_s)) * 1.0e15
    if (not np.isfinite(dt_min_fs)) or dt_min_fs <= 0.0:
        return np.arange(nframe, dtype=np.int64)

    selected = [0]
    last_t = float(time_fs[0])

    for i in range(1, nframe - 1):
        t = float(time_fs[i])
        if (t - last_t) >= dt_min_fs:
            selected.append(i)
            last_t = t

    if selected[-1] != (nframe - 1):
        selected.append(nframe - 1)

    return np.asarray(selected, dtype=np.int64)


def plot_animation_from_arrays(wave, psi_ds, kprob_ds, time_fs, x, pot, k_axis, ewave, interval_fs, time_resolution_log10_s=-13, run_label=None):
    nx = min(len(x), len(pot))
    if nx < 3:
        return

    x = x[:nx]
    pot = pot[:nx]
    wave = wave[:, :nx]
    psi_ds = psi_ds[:, :nx]

    nframe_all = min(wave.shape[0], psi_ds.shape[0], kprob_ds.shape[0], len(time_fs))
    if nframe_all < 1:
        return

    wave = wave[:nframe_all]
    psi_ds = psi_ds[:nframe_all]
    kprob_ds = kprob_ds[:nframe_all]
    time_fs = time_fs[:nframe_all]

    frame_indices = _frame_indices_from_time_resolution(time_fs, time_resolution_log10_s)
    if len(frame_indices) < nframe_all:
        wave = wave[frame_indices]
        psi_ds = psi_ds[frame_indices]
        kprob_ds = kprob_ds[frame_indices]
        time_fs = time_fs[frame_indices]

    nframe = len(time_fs)
    title_prefix = f"{run_label} | " if run_label else ""
    dt_min_fs = (10.0 ** float(time_resolution_log10_s)) * 1.0e15
    print(f"Animation frames: {nframe} / {nframe_all} (dt_min={dt_min_fs:.6g} fs, log10_s={int(time_resolution_log10_s)})")

    ix0 = 1
    ix1 = max(2, nx - 1)

    x_plot = x[ix0:ix1]
    pot_plot = pot[ix0:ix1] * har2ev
    wave_x_plot = wave[:, ix0:ix1]
    nk = min(len(k_axis), kprob_ds.shape[1])
    if nk < 3:
        return
    k_plot = np.asarray(k_axis[:nk], dtype=np.float32)
    wave_k_plot = np.asarray(kprob_ds[:, :nk], dtype=np.float32)

    pot_max = max(float(np.max(pot_plot)), 1e-6)
    wave_x_max = max(float(np.max(wave_x_plot)), 1e-6)
    wave_k_max = max(float(np.max(wave_k_plot)), 1e-6)

    effective_interval_fs = interval_fs if nframe < 2 else float(np.mean(np.diff(time_fs)))
    frame_duration_ms = max(1, int(effective_interval_fs * 1000.0))
    frame_duration_fast_ms = max(1, int(frame_duration_ms * 0.5))
    frame_duration_slow_ms = max(1, int(frame_duration_ms * 2.0))

    frames = [
        go.Frame(
            data=[
                go.Scatter(y=wave_x_plot[iframe]),
                go.Scatter(y=wave_k_plot[iframe])
            ],
            traces=[0, 2],
            name=str(iframe),
            layout=go.Layout(title_text=f'{title_prefix}Time = {time_fs[iframe]:.2f} fs')
        )
        for iframe in range(nframe)
    ]

    slider_steps = [
        {
            "args": [[str(iframe)], {"mode": "immediate", "frame": {"duration": 0, "redraw": True}, "transition": {"duration": 0}}],
            "label": f"{time_fs[iframe]:.2f}",
            "method": "animate"
        }
        for iframe in range(nframe)
    ]

    fig = make_subplots(
        rows=2,
        cols=1,
        specs=[[{"secondary_y": True}], [{}]],
        vertical_spacing=0.14,
        subplot_titles=('Real Space', 'k Space')
    )

    fig.add_trace(go.Scatter(x=x_plot, y=wave_x_plot[0], mode='lines', name='Wave Packet |psi|^2'), row=1, col=1, secondary_y=True)
    fig.add_trace(go.Scatter(x=x_plot, y=pot_plot, mode='lines', name='Potential'), row=1, col=1, secondary_y=False)
    fig.add_trace(go.Scatter(x=k_plot, y=wave_k_plot[0], mode='lines', name='k-space |psi(k)|^2'), row=2, col=1)

    fig.update_xaxes(title_text='Position [Angstrom]', row=1, col=1)
    fig.update_xaxes(title_text='k [1/Angstrom]', row=2, col=1)

    fig.update_yaxes(title_text='Energy [eV]', range=[0, pot_max * 1.2], row=1, col=1, secondary_y=False)
    fig.update_yaxes(title_text='Probability Density', range=[0, wave_x_max * 1.2], row=1, col=1, secondary_y=True)
    fig.update_yaxes(title_text='Probability Density', range=[0, wave_k_max * 1.2], row=2, col=1)

    fig.update_layout(
        width=850,
        height=820,
        template='plotly_white',
        title=f'{title_prefix}Time = {time_fs[0]:.2f} fs',
        margin={"b": 150},
        updatemenus=[
            {
                "type": 'buttons', "direction": 'left', "showactive": True, "active": 1,
                "x": 0.0, "y": 0.0, "xanchor": 'left', "yanchor": 'top', "pad": {"r": 6, "t": 30},
                "buttons": [
                    {"label": 'Play', "method": 'animate', "args": [None, _animate_args(frame_duration_ms)]},
                    {"label": 'Pause', "method": 'animate', "args": [[None], {"mode": "immediate", "frame": {"duration": 0, "redraw": False}, "transition": {"duration": 0}}]}
                ]
            },
            {
                "type": 'buttons', "direction": 'left', "showactive": True, "active": 1,
                "x": 0.0, "y": -0.12, "xanchor": 'left', "yanchor": 'top', "pad": {"r": 8, "t": 0},
                "buttons": [
                    {"label": '0.5x', "method": 'animate', "args": [None, _animate_args(frame_duration_slow_ms)]},
                    {"label": '1x', "method": 'animate', "args": [None, _animate_args(frame_duration_ms)]},
                    {"label": '2x', "method": 'animate', "args": [None, _animate_args(frame_duration_fast_ms)]}
                ]
            }
        ],
        sliders=[
            {
                "active": 0,
                "x": 0.22,
                "y": 0.0,
                "len": 0.76,
                "currentvalue": {"prefix": "Time [fs]: "},
                "pad": {"t": 30},
                "steps": slider_steps
            }
        ]
    )

    fig.frames = frames
    display(HTML(fig.to_html(full_html=False, include_plotlyjs='inline', auto_play=False)))

In [14]:
###############################################################################
# Interactive Widget
###############################################################################

# Base simulation controls
# Runtime grid/time variables are computed inside interactive().

widget_width = '600px'
widget_description_width = '260px'
widget_style = {'description_width': widget_description_width}


def make_widget_layout(display=''):
    return widgets.Layout(width=widget_width, display=display)


def make_dropdown(options, value, description):
    return widgets.Dropdown(options=options, value=value, description=description, style=widget_style, layout=make_widget_layout())


def make_int_slider(value, min, max, step, description, readout_format='d'):
    return widgets.IntSlider(value=value, min=min, max=max, step=step, description=description, continuous_update=False, orientation='horizontal', readout=True, readout_format=readout_format, style=widget_style, layout=make_widget_layout())


def make_float_slider(value, min, max, step, description, readout_format='.2f'):
    return widgets.FloatSlider(value=value, min=min, max=max, step=step, description=description, continuous_update=False, orientation='horizontal', readout=True, readout_format=readout_format, style=widget_style, layout=make_widget_layout())


pot_shape_widget = make_dropdown(
    options=[('Zero Potential', 0),
             ('Step Potential', 1),
             ('Single Potential Barrier', 2),
             ('Double Potential Barrier', 3),
             ('Square Potential well', 4),
             ('Harmonic Potential well', 5),
             ('Triangular Potential well', 6)],
    value=0,
    description='Shape of Potential:'
)

lpacket_widget = make_dropdown(
    options=[('wave packet', 0),
             ('linear combination', 1)],
    value=0,
    description='packet list:'
)


ewave_widget        = make_float_slider(value=10, min=0.1, max=100, step=0.001,
                                        description='Energy of Wavepacket [eV]:',
                                        readout_format='.3f')
curvature_widget    = make_float_slider(value=0.4, min=0.1, max=1.0, step=0.1,
                                        description='Curvature:',
                                        readout_format='.1f')
slope_widget        = make_float_slider(value=1.0, min=0.1, max=5.0, step=0.1,
                                        description='Slope: gentle <--> steep',
                                        readout_format='.1f')
pot_height_eV_widget        = make_float_slider(value=10, min=0.1, max=100, step=1,
                                        description='Potential Height [eV]:',
                                        readout_format='.2f')
width_widget        = make_float_slider(  value=10, min=2, max=200, step=0.1,
                                        description='Width [Angstrom]:')
barrier_distance_widget     = make_float_slider(  value=SIM_CFG.barrier_distance_ang, min=1, max=100, step=0.1,
                                                description='Barrier distance [Angstrom]:')
simulation_time_fs_widget   = make_float_slider(value=SIM_CFG.simulation_time_fs_default, min=1.0, max=50000, step=0.5,
                                                description='Simulation Time [fs]:',
                                                readout_format='.1f')
ncombistates_widget = make_int_slider(  value=10, min=1, max=500, step=1,
                                        description='Number of N basis:')
dispersion_gaussian_log10_widget = make_float_slider( value=1, min=-1, max=3, step=0.1,
                                                    description='Gaussian Dispersion (log) [Angstrom]:')

x_resolution_widget = make_dropdown(options=[('Low', 'low'), ('Mid', 'mid'), ('High', 'high'), ('Full', 'full')],
                                    value='mid',
                                    description='x resolution:')
time_resolution_widget  = make_dropdown(options=[('10^-16 s', -16), ('10^-15 s', -15), ('10^-14 s', -14), ('10^-13 s', -13), ('10^-12 s', -12), ('10^-11 s', -11)],
                                       value=-15,
                                       description='time resolution [s]:')


def set_widget_visible(widget, visible):
    widget.layout = make_widget_layout(display='' if visible else 'none')


def update_visibility(change):
    value = pot_shape_widget.value
    is_harmonic = (value == 5)
    set_widget_visible(curvature_widget, is_harmonic)
    set_widget_visible(width_widget, value in [2, 3, 4])
    set_widget_visible(barrier_distance_widget, value == 3)
    set_widget_visible(slope_widget, value in [6])
    set_widget_visible(lpacket_widget, is_harmonic)

    if not is_harmonic and lpacket_widget.value != 0:
        lpacket_widget.value = 0

    set_widget_visible(ncombistates_widget, is_harmonic and (lpacket_widget.value == 1))


def update_packet_visibility(change):
    is_harmonic = (pot_shape_widget.value == 5)
    set_widget_visible(ncombistates_widget, is_harmonic and (lpacket_widget.value == 1))


set_widget_visible(curvature_widget, False)
set_widget_visible(barrier_distance_widget, False)
set_widget_visible(slope_widget, False)
set_widget_visible(lpacket_widget, False)
set_widget_visible(ncombistates_widget, False)
pot_shape_widget.observe(update_visibility, names='value')
lpacket_widget.observe(update_packet_visibility, names='value')


@interact_manual(
    pot_shape=pot_shape_widget,
    pot_height_eV=pot_height_eV_widget,
    ewave=ewave_widget,
    dispersion_gaussian_log10=dispersion_gaussian_log10_widget,
    x_resolution=x_resolution_widget,
    time_resolution=time_resolution_widget,
    curvature=curvature_widget,
    width=width_widget,
    barrier_distance=barrier_distance_widget,
    slope=slope_widget,
    simulation_time_fs=simulation_time_fs_widget,
    lpacket=lpacket_widget,
    ncombistates=ncombistates_widget,
)
def interactive(pot_shape, pot_height_eV, ewave, dispersion_gaussian_log10, x_resolution, time_resolution, curvature, width, barrier_distance, slope, simulation_time_fs, lpacket, ncombistates):
    """
    Solve TDSE and display notebook animation.

    Notes:
    - Physical time step (dt_fs) is controlled in code via SIM_CFG.dt_fs.
    - time_resolution is log10 seconds for rendering frame spacing only.
    """
    dt_fs_local = float(SIM_CFG.dt_fs)
    dt_local = dt_fs_local / au2fs_time
    ngridt = max(1, int(np.ceil(float(simulation_time_fs) / dt_fs_local)))

    sigma_x_ang, boxL_run, ngridx_run, dx_ang_run = set_grid_by_dispersion(
        dispersion_gaussian_log10,
        SIM_CFG.dx_target_ang,
        SIM_CFG.ngridx_max,
    )
    boxL_bohr_run = boxL_run * ang2bohr
    dx_run = np.float64(dx_ang_run * ang2bohr)
    packet_center = set_packet_center(boxL_run, sigma_x_ang)

    if (pot_shape == 5) and (lpacket == 1):
        run_label = f"Mode: linear combination (N basis = {int(ncombistates)})"
    else:
        run_label = "Mode: wave packet"

    print(run_label)
    print(f"dt = {dt_fs_local:.3f} fs, steps = {ngridt}, simulated time ~= {ngridt * dt_fs_local:.3f} fs")
    print(f"boxL = {boxL_run:.2f} Angstrom, ngridx = {ngridx_run}, dx = {dx_ang_run:.4f} Angstrom")

    pot, wave_ds, psi_ds, kprob_ds, time_fs_ds, x_ang_ds, k_axis_ds, _t_stride, x_stride, k_stride, _reflection, _transmission = time_dep_schr_eq(
        pot_shape, pot_height_eV, ewave, dispersion_gaussian_log10, ngridt, lpacket, ncombistates,
        packet_center, barrier_distance, curvature, width, slope,
        boxL_run, boxL_bohr_run, ngridx_run, dx_run, dt_local,
        x_resolution, SIM_CFG.dx_target_ang
    )

    print(f"[k-space] source = full-grid FFT, x_resolution='{x_resolution}', x_stride={x_stride}, k_stride={k_stride}, k_bins={len(k_axis_ds)}")

    plot_animation_from_arrays(
        wave_ds,
        psi_ds,
        kprob_ds,
        time_fs_ds,
        x_ang_ds,
        pot[::x_stride],
        k_axis_ds,
        ewave,
        interval_fs=dt_fs_local,
        time_resolution_log10_s=time_resolution,
        run_label=run_label,
    )


update_visibility(None)
update_packet_visibility(None)

interactive(children=(Dropdown(description='Shape of Potential:', layout=Layout(display='', width='600px'), op…